In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betaln
from itertools import combinations
import yaml
import matplotlib.patches as mpatches
import yaml

with open("cfg.yml", "r") as f:
    cfg = yaml.safe_load(f)
with open("cfg.yml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f) or {}

for key, value in CFG.items():
    globals()[key] = value

NET_SIZES = cfg["net_size"]                         # 10, 25, 50, 100, 200
TOPOLOGIES = cfg["topology"]                        # "cycle", "wheel", ["clustered", "scale_free", "random"]
THEORIES = cfg["theories"]                          # [0.82, 0.81, 0.80, 0.79, 0.78], [0.9, 0.85, 0.8, 0.75, 0.7], [0.82, 0.81, 0.80, 0.70, 0.50] 
TOP_PRIORS_STRENGTH = cfg["top_priors_strength"]    # 4, 40, 400, 4000
DENSITIES = cfg["density"]                          # 0 to 1, step 0.1
STEPS = 100
N_RUNS = 5
SEED = 1

RNG = np.random.default_rng(SEED)

In [2]:
import pickle

with open("graphs_42.pkl", "rb") as f:
    graphs_42 = pickle.load(f)
with open("graphs_666.pkl", "rb") as f:
    graphs_666 = pickle.load(f)
with open("graphs_100.pkl", "rb") as f:
    graphs_100 = pickle.load(f)
with open("graphs_1.pkl", "rb") as f:
    graphs_1 = pickle.load(f)
with open("graphs_808.pkl", "rb") as f:
    graphs_808 = pickle.load(f)

In [3]:
graphs_42.keys()

dict_keys([10, 25, 50, 100, 200])

In [ ]:
''' 
Here I am setting some boilerplate functions for metrics as suggested by Claude. 
TODO: This cell will be filled in with more functions for metrics as I implement them.
'''

def beta_variance(alpha: float, beta: float) -> float:
    """Variance of Beta(alpha, beta)."""
    s = alpha + beta
    return (alpha * beta) / (s**2 * (s + 1))
 
from scipy.stats import beta as beta_dist

def beta_entropy(alpha: float, beta: float) -> float:
    """Differential entropy of Beta(alpha, beta)."""
    return beta_dist.entropy(alpha, beta)
 
def kl_beta(a1, b1, a2, b2) -> float:
    """KL divergence KL(Beta(a1,b1) || Beta(a2,b2))."""
    from scipy.special import digamma
    s1, s2 = a1 + b1, a2 + b2
    return (betaln(a2, b2) - betaln(a1, b1)
            + (a1 - a2) * digamma(a1)
            + (b1 - b2) * digamma(b1)
            + (s2 - s1) * digamma(s1))
 
def mean_pairwise_kl(params: np.ndarray) -> float:
    """Average pairwise KL divergence across all node pairs (symmetrised)."""
    n = len(params)
    if n < 2:
        return 0.0
    total, count = 0.0, 0
    for i, j in combinations(range(n), 2):
        a1, b1 = params[i]
        a2, b2 = params[j]
        kl_sym = 0.5 * (kl_beta(a1, b1, a2, b2) + kl_beta(a2, b2, a1, b1))
        total += kl_sym
        count += 1
    return total / count


In [ ]:
def compute_metrics(history: list[np.ndarray]) -> dict:
    """Return time-series of mean variance, mean KL, and belief mean std."""
    mean_vars, mean_kls, belief_stds = [], [], []
    for params in history:
        variances = [beta_variance(p[0], p[1]) for p in params]
        mean_vars.append(np.mean(variances))
        mean_kls.append(mean_pairwise_kl(params))
        means = params[:, 0] / params.sum(axis=1)
        belief_stds.append(np.std(means))
    return {
        "mean_variance": np.array(mean_vars),
        "mean_kl":       np.array(mean_kls),
        "belief_std":    np.array(belief_stds),
    }


def simulate(graph: nx.Graph, steps: int = 100):
    
    params = None                               # TODO: These will be the evolving params for the nodes in the graph
    history = [params.copy()]                   # Start history tracking
    nodes = list(graph.nodes())                 # Get nodes
 
    for _ in range(steps):
        new_params = params.copy()
        
        # ========== TODO: Do some stuff ========== #

        params = new_params
        history.append(params.copy())
 
    return history

# Example usage for one single run
history = simulate(graphs_42["cycle"], steps=STEPS)

metrics = compute_metrics(history)